# Data Profile & Data Quality 파이프라인

이 노트북은 Google Cloud의 Knowledge Catalog DataScan API를 사용하여 `thelook_ecommerce` 데이터셋에 속한 주요 테이블들의 **데이터 프로파일(Data Profile)** 및 **데이터 품질(Data Quality)** 스캔을 자동 생성, 실행하고 결과를 모니터링합니다.

### 학습 목표
1. **데이터 프로파일(Data Profile)**: 테이블의 각 컬럼 분포 및 통계치를 분석하고, 이를 BigQuery 스키마 메타데이터와 동기화하는 방법을 이해합니다.
2. **데이터 품질(Data Quality)**: BigQuery 테이블 스키마 분석을 바탕으로 동적 품질 검사 규칙(Dynamic Rule)을 생성하고 스캔을 실행합니다.
3. **비동기 일괄 파이프라인**: 대량의 테이블을 대상으로 DataScan 작업을 비동기 병렬로 가동하고 진행 상태를 모니터링하는 코드를 구현합니다.

### 작업 파이프라인

1. **환경 초기화**: BigQuery 클라이언트를 초기화하고 REST 호출을 위한 OAuth2 토큰을 발급받습니다.
2. **API 호출 공통 함수 정의**: 지수 백오프 기반의 재시도 기능이 내장된 REST API 호출 헬퍼 및 비동기 Job 상태 모니터링 유틸리티를 정의합니다.
3. **데이터 프로파일 및 품질 스캔 생성**: Knowledge Catalog API를 활용하여 테이블별 데이터 프로파일 스캔 및 동적 룰(Dynamic Rule) 기반 데이터 품질 검사 스캔을 자동 구성합니다.
4. **단일 테이블 테스트 검증**: `users` 테이블을 활용하여 데이터 프로파일 및 품질 상태를 시뮬레이션하고, 빅쿼리 내 적재된 데이터 스캔 결과 테이블을 조회하여 분석 상태를 확인합니다.
5. **일괄 적용 및 모니터링 파이프라인**: 모든 대상 테이블의 스캔을 병렬 기동하고, 완료된 테이블마다 데이터 분류 및 스캔 메타데이터 퍼블리싱을 연동하는 공식 라벨을 부착합니다.

## Step 1: 초기 환경 설정 및 REST API 설정

In [ ]:
import json
import time
import urllib.request
import urllib.error
import ssl
import google.auth
from google.auth.transport.requests import Request
from google.cloud import bigquery

# 1. GCP 인증 및 기본 프로젝트 정보 로드 (Application Default Credentials 사용)
credentials, PROJECT_ID = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])

DATASET_ID = "thelook_ecommerce"  # 분석 대상 빅쿼리 데이터셋명
LOCATION = "us-central1"      # Knowledge Catalog DataScan이 실행 및 생성될 GCP 리전 위치 (us-central1 리전)

# 2. Knowledge Catalog REST API 호출용 OAuth2 Access Token 발급
credentials.refresh(Request())
ACCESS_TOKEN = credentials.token

# 인증서 확인 우회를 위한 SSL 컨텍스트 설정 (테스트 및 실습 목적)
ssl_context = ssl._create_unverified_context()

# BigQuery API 호출용 공식 Python SDK 클라이언트 초기화
bq_client = bigquery.Client(project=PROJECT_ID, credentials=credentials)

print(f"현재 인식된 Project ID: {PROJECT_ID}, 대상 Dataset: {DATASET_ID}")
print("GCP 환경 및 API 인증 설정 완료!")

## Step 2: API 호출 공통 함수 정의

In [ ]:
# [공통 유틸리티] Knowledge Catalog REST API 전송 함수
def make_rest_request(url, method="GET", body_dict=None, max_retries=5):
    """
    OAuth2 Bearer 토큰 및 SSL 우회 방식을 적용하여 보안 REST API 통신을 수행합니다.
    """
    req = urllib.request.Request(url, method=method)
    # 1단계에서 발급받은 OAuth2 임시 토큰을 Authorization Bearer 헤더에 주입하여 API 인증을 통과합니다.
    req.add_header("Authorization", f"Bearer {ACCESS_TOKEN}")
    req.add_header("Content-Type", "application/json")
    
    data = None
    if body_dict:
        data = json.dumps(body_dict).encode("utf-8")
        
    retries = 0
    backoff = 2  # 시작 대기 시간 (초)
    
    while True:
        try:
            with urllib.request.urlopen(req, data=data, context=ssl_context) as response:
                return json.loads(response.read().decode("utf-8"))
        except urllib.error.HTTPError as e:
            err_msg = e.read().decode("utf-8")
            # 429 Quota/Rate Limit 에러 시 지수 백오프 기반 재시도 처리
            if e.code == 429 and retries < max_retries:
                print(f"  [429 Quota Exceeded] {backoff}초 후 재시도 합니다 (재시도: {retries + 1}/{max_retries})...")
                time.sleep(backoff)
                retries += 1
                backoff *= 2
                continue
            # API 호출 중 에러가 발생한 경우 상세 에러 응답 바디를 파싱하여 예외를 발생시킵니다.
            raise Exception(f"HTTP Error {e.code} - {err_msg}")

# [공통 유틸리티] DataScan Job 비동기 기동 및 대기 함수
def run_datascan_and_wait(scan_id):
    """
    기동된 데이터 스캔(DataScan) Job이 완료(SUCCEEDED)될 때까지 10초 주기로 진행 상태를 폴링(Polling)하며, 완료 후 전체 상세 분석 정보(?view=FULL)가 담긴 결과를 조회해 반환합니다.
    """
    # 1. 특정 DataScan을 수동으로 일회성 실행(:run)하기 위한 POST API 요청 URL 정의
    run_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}:run"
    print(f"  -> DataScan 실행 요청 중 (Scan ID: {scan_id})...")
    run_res = make_rest_request(run_url, method="POST")
    
    # 2. 기동된 비동기 Job의 이름(프로젝트, 리전, 스캔ID 및 고유 Job ID 조합) 파싱
    job_name = run_res["job"]["name"]
    job_id = run_res["job"]["uid"]
    print(f"  -> 실행 Job ID: {job_id} (완료될 때까지 상태를 폴링하기 시작합니다...)")
    
    # 3. 비동기 백엔드 작업 진행률 모니터링을 위한 상태 폴링 루프 (?view=FULL 파라미터 추가하여 상세 분석 결과까지 로드)
    job_url = f"https://dataplex.googleapis.com/v1/{job_name}?view=FULL"
    while True:
        job = make_rest_request(job_url, method="GET")
        state = job.get("state")
        print(f"     [폴링] 현재 작업 상태: {state}")
        
        if state == "SUCCEEDED":
            print("  -> 스캔 완료!")
            return job
        elif state in ["FAILED", "CANCELLED"]:
            raise Exception(f"DataScan Job 실행 오류 발생 (상태: {state})")
            
        # 10초 대기 후 재조회 (Google Cloud API의 Rate Limit을 준수하기 위한 간격)
        time.sleep(10)

# [공통 유틸리티] DataScan Job 비동기 기동 함수 (대기 없음)
def trigger_datascan(scan_id):
    """
    정의된 DataScan의 실행 Job을 구동하고 완료될 때까지 대기하지 않고 Job 이름을 반환합니다.
    """
    run_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}:run"
    run_res = make_rest_request(run_url, method="POST")
    return run_res["job"]["name"]

# [공통 유틸리티] 비동기 데이터 스캔 Job 진행 모니터링 및 완료 시 테이블 라벨 매핑 처리 함수
def monitor_datascan_jobs_and_label(active_jobs, poll_interval_seconds=30):
    """
    비동기로 기동된 Knowledge Catalog DataScan Job 목록(active_jobs)을 실시간 폴링 모니터링합니다.
    각 작업이 완료(SUCCEEDED)되면 대상 빅쿼리 테이블에 스캔 메타데이터 퍼블리싱 라벨을 자동으로 매핑합니다.
    """
    total_triggered = len(active_jobs)
    print(f"\n총 {total_triggered}개의 데이터 스캔 작업이 동시에 기동되었습니다. 진행 상황 모니터링을 시작합니다.")
    print(f"({poll_interval_seconds}초 주기로 전체 현황을 요약 출력하며 완료 시 라벨을 테이블에 바인딩합니다.)\n")

    while active_jobs:
        status_summary = {"PENDING": 0, "RUNNING": 0, "SUCCEEDED": 0, "FAILED": 0, "CANCELLED": 0}
        
        # 작업 상태 확인 및 처리 완료 대상 식별
        for job_name in list(active_jobs.keys()):
            job_url = f"https://dataplex.googleapis.com/v1/{job_name}"
            try:
                job = make_rest_request(job_url, method="GET")
                state = job.get("state", "UNKNOWN")
                
                status_summary[state] = status_summary.get(state, 0) + 1
                
                if state in ["SUCCEEDED", "FAILED", "CANCELLED"]:
                    t_id, s_type = active_jobs[job_name]
                    print(f"  [완료] '{t_id}' {s_type} 작업 종료 (결과: {state})")
                    
                    # 스캔 성공 완료 시 빅쿼리 테이블 메타데이터 퍼블리싱 라벨 매핑 추가 (dataplex- 공식 접두사 적용)
                    if state == "SUCCEEDED":
                        try:
                            table_ref = bq_client.dataset(DATASET_ID).table(t_id)
                            table = bq_client.get_table(table_ref)
                            
                            labels = dict(table.labels or {})
                            scan_id = f"dp-thelook-{t_id}".lower().replace("_", "-") if s_type == "Profile" else f"dq-thelook-{t_id}".lower().replace("_", "-")
                            label_key = "dataplex-data-profile-published-scan" if s_type == "Profile" else "dataplex-data-quality-published-scan"
                            
                            labels[label_key] = scan_id
                            labels[f"dataplex-data-{s_type.lower()}-published-project"] = PROJECT_ID
                            labels[f"dataplex-data-{s_type.lower()}-published-location"] = LOCATION
                            
                            table.labels = labels
                            bq_client.update_table(table, ["labels"])
                            print(f"    -> '{t_id}' 테이블에 {s_type} 퍼블리싱 라벨 매핑 성공!")
                        except Exception as le:
                            print(f"    -> '{t_id}' 테이블 라벨 매핑 오류: {le}")
                    
                    del active_jobs[job_name]
            except Exception as e:
                # 일시적 오류는 다음 회차에 재시도하도록 패스
                pass
                
        if active_jobs:
            running_cnt = status_summary.get("RUNNING", 0) + status_summary.get("PENDING", 0)
            succeeded_cnt = total_triggered - len(active_jobs) - status_summary.get("FAILED", 0) - status_summary.get("CANCELLED", 0)
            
            print(f"[진행 상태] 진행 중: {running_cnt}개 | 완료: {succeeded_cnt}개 | 실패: {status_summary.get('FAILED', 0)}개", end="\r")
            time.sleep(poll_interval_seconds)

    print("\n\n=== [모든 테이블 데이터 스캔 모니터링 및 라벨 매핑 프로세스 완료] ===")

## Step 3: 데이터 프로파일 및 품질 스캔 생성

In [ ]:
# [Data Profile] 데이터 프로파일 스캔 조회 및 생성 함수
def get_or_create_data_profile_scan(table_id, sampling_percent=10.0):
    """
    대상 테이블에 대한 DATA_PROFILE Knowledge Catalog DataScan 리소스를 조회하고, 없으면 생성합니다.
    검증 결과는 지정된 BigQuery 데이터셋 내의 'datascan_profile_results' 테이블로 내보냅니다.
    """
    scan_id = f"dp-thelook-{table_id}".lower().replace("_", "-")
    get_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}"
    
    try:
        scan = make_rest_request(get_url, method="GET")
        print(f"  -> 기존 Data Profile Scan 발견: {scan_id}")
        return scan_id
    except Exception as e:
        if "404" in str(e) or "does not exist" in str(e):
            print(f"  -> 새로운 Data Profile Scan 생성 중: {scan_id}...")
            create_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans?dataScanId={scan_id}"
            
            export_table = f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/datascan_profile_results"
            
            # DataScan 유형을 'DATA_PROFILE'로 정의하고 프로파일 옵션 주입
            body = {
                "data": {
                    "resource": f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/{table_id}"
                },
                "executionSpec": {
                    "trigger": { "onDemand": {} }
                },
                "type": "DATA_PROFILE",
                "dataProfileSpec": {
                    "samplingPercent": sampling_percent, # 분석 효율성 및 비용 제어를 위해 샘플링 비율 설정
                    "postScanActions": {
                        "bigqueryExport": {
                            # 분석 완료된 프로파일 보고서 로우 데이터를 빅쿼리 테이블로 내보내는 동작 정의
                            "resultsTable": export_table
                        }
                    },
                    "catalogPublishingEnabled": True
                }
            }
            operation = make_rest_request(create_url, method="POST", body_dict=body)
            op_name = operation["name"]
            
            while True:
                op_status = make_rest_request(f"https://dataplex.googleapis.com/v1/{op_name}", method="GET")
                if op_status.get("done"):
                    if "error" in op_status:
                        raise Exception(f"Data Profile Scan 생성 실패: {op_status['error']}")
                    break
                time.sleep(5)
            print(f"  -> Data Profile Scan 생성 완료: {scan_id}")
            return scan_id
        else:
            raise e

### 데이터 품질 스캔 정의 및 규칙 구성

데이터 품질 스캔은 스키마에 정의된 제약조건(예: `NOT NULL`, 가격이 `0`보다 큰지 여부)을 평가하기 위해 동적 규칙(Dynamic Quality Rules)을 자동으로 빌드하고 모니터링합니다.

마찬가지로 리소스 구조를 구성(Create)하거나 이미 존재하는 경우 재사용하는 방식으로 동작합니다.

In [ ]:
# [Data Quality] 빅쿼리 스키마 분석 기반 동적 품질 규칙(Rule) 생성기
def generate_default_quality_rules(table_id):
    """
    빅쿼리 테이블의 실제 스키마 정보를 실시간으로 읽어와 논리적 기본 데이터 품질 규칙(Rule)을 동적으로 구성합니다.
    이를 통해 하드코딩 없이 테이블 스펙 변화에 유연하게 대응합니다.
    """
    table_ref = bq_client.dataset(DATASET_ID).table(table_id)
    table = bq_client.get_table(table_ref)
    
    # 범위 검색이 필요한 컬럼(VALIDITY 차원) 설정을 사전에 매핑 정의 (확장성 확보)
    VALID_RANGES = {
        "latitude": {
            "min": "-90", 
            "max": "90", 
            "desc": "위도 좌표는 유효 범위(-90 ~ 90) 내에 있어야 합니다."
        },
        "longitude": {
            "min": "-180", 
            "max": "180", 
            "desc": "경도 좌표는 유효 범위(-180 ~ 180) 내에 있어야 합니다."
        }
    }
    
    rules = []
    
    # 테이블 내의 컬럼 정보 리스트를 순회하며 적합한 규칙 정의
    for field in table.schema:
        field_type = field.field_type.upper()
        
        # 1. COMPLETENESS (완전성) 차원: REQUIRED(필수값) 컬럼은 NULL 값을 허용하지 않는 규칙 추가
        if field.mode == "REQUIRED":
            rules.append({
                "column": field.name,
                "dimension": "COMPLETENESS",
                "nonNullExpectation": {}, # NULL 불가 조건 설정
                "description": f"{field.name} 컬럼은 필수값이므로 NULL을 허용하지 않습니다."
            })
            
        # 2. UNIQUENESS (유일성) 차원: 컬럼명이 고유 키인 'id'일 경우 중복값을 허용하지 않는 규칙 추가
        if field.name == "id":
            rules.append({
                "column": field.name,
                "dimension": "UNIQUENESS",
                "uniquenessExpectation": {}, # 중복 불가 조건 설정
                "description": f"{field.name} 컬럼은 고유 레코드 식별자이므로 중복을 허용하지 않습니다."
            })
            
        # 3. VALIDITY (유효성) 차원: 수치형 좌표 컬럼의 범위 한계 검증 (매핑 테이블 기반 처리)
        if field.name in VALID_RANGES and field_type in ("FLOAT", "INTEGER"):
            range_spec = VALID_RANGES[field.name]
            rules.append({
                "column": field.name,
                "dimension": "VALIDITY",
                "rangeExpectation": {
                    "minValue": range_spec["min"],
                    "maxValue": range_spec["max"]
                },
                "description": range_spec["desc"]
            })
            
    return rules

# [Data Quality] 데이터 품질 스캔 조회 및 생성 함수
def get_or_create_data_quality_scan(table_id, sampling_percent=100.0):
    """
    대상 테이블에 대한 DATA_QUALITY Knowledge Catalog DataScan 리소스를 조회하고, 없으면 생성합니다.
    검증 결과는 지정된 BigQuery 데이터셋 내의 'datascan_quality_results' 테이블로 내보냅니다.
    """
    scan_id = f"dq-thelook-{table_id}".lower().replace("_", "-")
    get_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans/{scan_id}"
    
    try:
        scan = make_rest_request(get_url, method="GET")
        print(f"  -> 기존 Data Quality Scan 발견: {scan_id}")
        return scan_id
    except Exception as e:
        if "404" in str(e) or "does not exist" in str(e):
            print(f"  -> 새로운 Data Quality Scan 생성 중: {scan_id}...")
            create_url = f"https://dataplex.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/dataScans?dataScanId={scan_id}"
            
            # 동적으로 데이터 품질 검사 규칙 리스트 확보
            rules = generate_default_quality_rules(table_id)
            if not rules:
                # 테이블에 마땅한 특수 규칙이 검출되지 않는 경우, 임시로 첫 번째 컬럼에 기본 NULL 체크 규칙을 심어 에러를 방지합니다.
                fields_list = [f.name for f in bq_client.get_table(bq_client.dataset(DATASET_ID).table(table_id)).schema]
                check_col = "id" if "id" in fields_list else bq_client.get_table(bq_client.dataset(DATASET_ID).table(table_id)).schema[0].name
                rules = [{
                    "column": check_col,
                    "dimension": "COMPLETENESS",
                    "nonNullExpectation": {},
                    "description": "임시 기본 NULL 방지 체크"
                }]
                
            export_table = f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/datascan_quality_results"
            
            # DataScan 유형을 'DATA_QUALITY'로 정의하고 동적 생성된 규칙 목록 주입
            body = {
                "data": {
                    "resource": f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/{table_id}"
                },
                "executionSpec": {
                    "trigger": { "onDemand": {} }
                },
                "type": "DATA_QUALITY",
                "dataQualitySpec": {
                    "rules": rules,
                    "samplingPercent": sampling_percent,
                    "postScanActions": {
                        "bigqueryExport": {
                            # 분석 완료된 프로파일 보고서 로우 데이터를 빅쿼리 테이블로 내보내는 동작 정의
                            "resultsTable": export_table
                        }
                    },
                    "catalogPublishingEnabled": True
                }
            }
            operation = make_rest_request(create_url, method="POST", body_dict=body)
            op_name = operation["name"]
            
            while True:
                op_status = make_rest_request(f"https://dataplex.googleapis.com/v1/{op_name}", method="GET")
                if op_status.get("done"):
                    if "error" in op_status:
                        raise Exception(f"Data Quality Scan 생성 실패: {op_status['error']}")
                    break
                time.sleep(5)
            print(f"  -> Data Quality Scan 생성 완료: {scan_id}")
            return scan_id
        else:
            raise e

## Step 4: 단일 테이블 테스트 검증

예시 테이블(`users`)을 통해 데이터 프로파일 및 품질 검사 스캔을 직접 호출하고 결과를 획득하여 정상 동작하는지 테스트 및 시뮬레이션을 진행합니다.

### A. 데이터 프로파일 테스트 (Data Profile Test)

사용자(`users`) 테이블을 샘플로 선택하여 데이터 프로파일링 파이프라인의 작동 여부를 검증합니다.

In [ ]:
TEST_TABLE = "users"

print(f"=== [{TEST_TABLE}] 데이터 프로파일 시뮬레이션 시작 ===")

# 데이터 프로파일 생성 및 비동기 실행 (데이터 비용 절감을 위해 10% 샘플링 기동)
dp_scan_id = get_or_create_data_profile_scan(TEST_TABLE, sampling_percent=10.0)
dp_job = run_datascan_and_wait(dp_scan_id) # Job 기동 후 완료 상태(SUCCEEDED)까지 폴링 대기

# 빅쿼리 테이블 객체를 가져와 데이터 프로파일 published 라벨 매핑 추가 (dataplex- 공식 접두사 적용)
try:
    table_ref = bq_client.dataset(DATASET_ID).table(TEST_TABLE)
    table = bq_client.get_table(table_ref)
    
    labels = dict(table.labels or {})
    labels["dataplex-data-profile-published-scan"] = dp_scan_id
    labels["dataplex-data-profile-published-project"] = PROJECT_ID
    labels["dataplex-data-profile-published-location"] = LOCATION
    table.labels = labels
    bq_client.update_table(table, ["labels"])
    print(f"  -> [{TEST_TABLE}] 테이블에 데이터 프로파일 퍼블리싱 라벨 적용 완료!")
except Exception as e:
    print(f"  -> [{TEST_TABLE}] 라벨 업데이트 실패: {e}")

### A-1. 빅쿼리에서 데이터 프로파일 결과 확인

데이터 프로파일 스캔의 상세 결과는 지정된 BigQuery 테이블(`datascan_profile_results`)에 자동으로 저장됩니다. 아래 쿼리를 실행하여 최근 완료된 프로파일 결과를 확인할 수 있습니다.

In [ ]:
# [BigQuery 조회] 최근 데이터 프로파일 결과 확인
query = f"""
SELECT 
    column_name,
    column_type,
    percent_null,
    percent_unique,
    min_value,
    max_value,
    average_value
FROM `{PROJECT_ID}.{DATASET_ID}.datascan_profile_results`
WHERE data_source.table_id = '{TEST_TABLE}'
  AND job_start_time = (
      SELECT MAX(job_start_time) 
      FROM `{PROJECT_ID}.{DATASET_ID}.datascan_profile_results`
      WHERE data_source.table_id = '{TEST_TABLE}'
  )
ORDER BY column_name
"""
print(f"Executing query to retrieve profile results for '{TEST_TABLE}'...")
df = bq_client.query(query).to_dataframe()
df

### B. 데이터 품질 테스트 (Data Quality Test)

동일한 사용자(`users`) 테이블에 대해 동적으로 감지된 품질 규칙(Rule)들을 적용하고 스캔을 실행하여 데이터 정합성 검증 파이프라인을 테스트합니다.

In [ ]:
TEST_TABLE = "users"

print(f"=== [{TEST_TABLE}] 데이터 품질 검사 시뮬레이션 시작 ===")

# 데이터 품질 검사 생성 및 비동기 실행 (데이터 비용 절감을 위해 10% 샘플링 기동)
dq_scan_id = get_or_create_data_quality_scan(TEST_TABLE, sampling_percent=10.0)
dq_job = run_datascan_and_wait(dq_scan_id) # Job 기동 후 완료 상태(SUCCEEDED)까지 폴링 대기

# 빅쿼리 테이블 객체를 가져와 데이터 품질 published 라벨 매핑 추가 (dataplex- 공식 접두사 적용)
try:
    table_ref = bq_client.dataset(DATASET_ID).table(TEST_TABLE)
    table = bq_client.get_table(table_ref)
    
    labels = dict(table.labels or {})
    labels["dataplex-data-quality-published-scan"] = dq_scan_id
    labels["dataplex-data-quality-published-project"] = PROJECT_ID
    labels["dataplex-data-quality-published-location"] = LOCATION
    table.labels = labels
    bq_client.update_table(table, ["labels"])
    print(f"  -> [{TEST_TABLE}] 테이블에 데이터 품질 퍼블리싱 라벨 적용 완료!")
except Exception as e:
    print(f"  -> [{TEST_TABLE}] 라벨 업데이트 실패: {e}")

### B-1. 빅쿼리에서 데이터 품질 검사 결과 확인

데이터 품질 검증 결과는 지정된 BigQuery 테이블(`datascan_quality_results`)로 자동 내보내기 됩니다. 다음 쿼리를 통해 최근에 수행된 품질 검사의 통과 여부 및 세부 규칙 결과를 모니터링할 수 있습니다.

In [ ]:
# [BigQuery 조회] 최근 데이터 품질 결과 확인
query = f"""
SELECT 
    rule_column,
    rule_dimension,
    rule_description,
    rule_passed,
    rule_rows_evaluated,
    rule_rows_passed,
    rule_rows_passed_percent
FROM `{PROJECT_ID}.{DATASET_ID}.datascan_quality_results`
WHERE data_source.table_id = '{TEST_TABLE}'
  AND job_start_time = (
      SELECT MAX(job_start_time) 
      FROM `{PROJECT_ID}.{DATASET_ID}.datascan_quality_results`
      WHERE data_source.table_id = '{TEST_TABLE}'
  )
ORDER BY rule_column
"""
print(f"Executing query to retrieve quality results for '{TEST_TABLE}'...")
df = bq_client.query(query).to_dataframe()
df

## Step 5: 일괄 적용 및 모니터링 파이프라인

모든 대상 테이블에 대해 데이터 프로파일 및 품질 스캔을 병렬(비동기)적으로 한꺼번에 가동하고, 완료된 테이블마다 데이터 분류 및 스캔 메타데이터 퍼블리싱을 연동하는 공식 라벨을 부착합니다.

In [ ]:
# [메인 실행 파이프라인] 전체 테이블 대상 일괄 메타데이터 구축 실행
# 전체 테이블들을 비동기 병렬 기동하고 요약 모니터링을 진행합니다.

ALL_TABLES = [
    # "users",
    "orders",
    "order_items",
    "products",
    "events",
    "inventory_items",
    "distribution_centers"
]

print("=== [전체 테이블 대상 데이터 프로파일 / 품질 스캔 자동화 비동기 일괄 기동] ===")

active_jobs = {} # {job_name: (table_id, scan_type)}

for t_id in ALL_TABLES:
    print(f"  -> '{t_id}' 테이블 스캔 기동 중...")
    try:
        # A. 프로파일 스캔 조회/생성 및 실행 (10% 샘플링 적용)
        dp_scan = get_or_create_data_profile_scan(t_id, sampling_percent=10.0)
        dp_job_name = trigger_datascan(dp_scan)
        active_jobs[dp_job_name] = (t_id, "Profile")
        
        # B. 품질 검증 스캔 조회/생성 및 실행 (비용 절감을 위해 10% 샘플링 적용)
        dq_scan = get_or_create_data_quality_scan(t_id, sampling_percent=10.0)
        dq_job_name = trigger_datascan(dq_scan)
        active_jobs[dq_job_name] = (t_id, "Quality")
        
        # API 할당량(Quota) 초과 방지를 위한 기동 지연 시간 추가
        time.sleep(1)
        
    except Exception as e:
        print(f"  [오류 발생] '{t_id}' 테이블 기동 실패: {e}")

# 공통 모니터링 및 라벨 자동화 함수를 호출하여 비동기 모니터링 프로세스를 가동합니다.
monitor_datascan_jobs_and_label(active_jobs, poll_interval_seconds=30)